In [1]:
import os
import random
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    log_loss,
    roc_auc_score,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
    brier_score_loss,
)

from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve


# =========================================================
# 1. SETTINGS
# =========================================================

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

DATA_FILE = "cleaned_full_panel.csv"
OUTPUT_DIR = "outputs"
FIG_DIR = "figures"

WARMUP_YEARS = 3

STRUCT_ALPHA = 0.50
STRUCT_K = 2.0
EPS = 1e-6

# Choose "y_true" for primary target or "y_alt" for alternative target.
TARGET_COL = "y_true"

# Calibration is supplementary; main table should use p_hybrid_cv.
USE_ISOTONIC = True
ISO_BLEND = 0.10

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)


# =========================================================
# 2. HELPERS
# =========================================================

def safe_prob(p, eps=EPS):
    return np.clip(np.asarray(p, dtype=float), eps, 1.0 - eps)


def safe_logit(p, eps=EPS):
    p = safe_prob(p, eps=eps)
    return np.log(p) - np.log1p(-p)


def sigmoid(z):
    return expit(z)


def valid_xy(df, y_col, p_col):
    tmp = df[[y_col, p_col]].dropna().copy()
    return tmp[y_col].astype(int).values, tmp[p_col].astype(float).values


def make_logit_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(
            penalty="l2",
            solver="lbfgs",
            max_iter=1000,
            random_state=RANDOM_STATE,
        ),
    )


def compare_models(oos, target_col="y_true", include_calibrated=False):
    rows = []
    y = oos[target_col].astype(int).values

    model_cols = [
        ("p_struct", "STRUCT"),
        ("p_logit", "LOGIT"),
        ("p_hybrid_cv", "HYBRID"),
        ("p_naive", "NAIVE"),
        ("p_lag", "LAG"),
    ]

    if include_calibrated:
        model_cols.insert(3, ("p_hybrid_cv_cal", "HYBRID_CAL"))

    for col, label in model_cols:
        if col not in oos.columns:
            continue

        p = safe_prob(oos[col].astype(float).values)
        mask = ~np.isnan(p) & ~np.isnan(y)

        if not np.any(mask):
            continue

        y_m = y[mask]
        p_m = p[mask]

        brier = np.mean((p_m - y_m) ** 2)
        rmse = np.sqrt(brier)
        ll = log_loss(y_m, p_m, labels=[0, 1])

        if np.unique(y_m).size > 1:
            auc_val = roc_auc_score(y_m, p_m)
            ap = average_precision_score(y_m, p_m)
        else:
            auc_val = 0.5
            ap = y_m.mean()

        rows.append([label, brier, ll, rmse, auc_val, ap, int(mask.sum())])

    out = pd.DataFrame(
        rows,
        columns=["Model", "Brier", "LogLoss", "RMSE", "AUC", "AP", "N"],
    )

    return out.sort_values("Brier").reset_index(drop=True)


def make_oos_dates(df, warmup=3):
    rows = []

    for c, g in df.groupby("country"):
        yrs = np.sort(g["year"].dropna().unique())

        if len(yrs) > warmup:
            rows.append(pd.DataFrame({"country": c, "year": yrs[warmup:]}))

    if rows:
        return pd.concat(rows, ignore_index=True)

    return pd.DataFrame(columns=["country", "year"])

# =========================================================
# 3. BENCHMARK MODELS
# =========================================================

def compute_naive_prob(df, country, year, target_col="y_true"):
    """
    NAIVE benchmark:
    Pooled historical prevalence using only information before forecast year.
    """
    hist = df[df["year"] < year].copy()

    if hist.empty:
        prior = df[target_col].mean()
    else:
        prior = hist[target_col].mean()

    if pd.isna(prior):
        prior = 0.5

    return float(safe_prob(prior))


def compute_lag_prob(df, country, year, target_col="y_true"):
    """
    LAG benchmark:
    p_lag_{it} = y_{i,t-1}, clipped only for numerical stability.
    """
    row = df[(df["country"] == country) & (df["year"] == year)]

    if row.empty:
        return np.nan

    prev_target = row["prev_target"].iloc[0]

    if pd.notna(prev_target):
        return float(safe_prob(prev_target))

    hist = df[df["year"] < year]
    prior = hist[target_col].mean()

    if pd.isna(prior):
        prior = 0.5

    return float(safe_prob(prior))

def bootstrap_auc_diff(y, p1, p2, n_boot=1000, random_state=42):
    """
    Pooled nonparametric bootstrap for AUC differences.
    """
    rng = np.random.default_rng(random_state)
    diffs = []
    n = len(y)

    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        y_b = y[idx]

        if len(np.unique(y_b)) < 2:
            continue

        auc1 = roc_auc_score(y_b, p1[idx])
        auc2 = roc_auc_score(y_b, p2[idx])
        diffs.append(auc1 - auc2)

    diffs = np.asarray(diffs)

    if len(diffs) == 0:
        return {
            "mean_diff": np.nan,
            "ci_lower": np.nan,
            "ci_upper": np.nan,
            "n_boot_used": 0,
        }

    return {
        "mean_diff": float(np.mean(diffs)),
        "ci_lower": float(np.percentile(diffs, 2.5)),
        "ci_upper": float(np.percentile(diffs, 97.5)),
        "n_boot_used": int(len(diffs)),
    }

# =========================================================
# 4. LOAD CLEANED REPLICATION DATASET
# =========================================================

panel = pd.read_csv(DATA_FILE)

required_cols = [
    "country", "year",
    "EAR", "ROA", "LLR", "LIQ", "WFS",
    "frag_capital", "frag_profit", "frag_assetq", "frag_liquidity",
    "stress_score", "y_true", "y_alt",
    "EAR_lag1", "ROA_lag1", "LLR_lag1", "LIQ_lag1",
    "prev_y",
]

missing_cols = [c for c in required_cols if c not in panel.columns]

if missing_cols:
    raise ValueError(f"Missing required columns in {DATA_FILE}: {missing_cols}")

panel["year"] = pd.to_numeric(panel["year"], errors="coerce").astype(int)
panel = panel.sort_values(["country", "year"]).reset_index(drop=True)

print("Loaded replication dataset:", DATA_FILE)
print("Panel shape:", panel.shape)
print("Countries:", sorted(panel["country"].unique()))
print("Years:", int(panel["year"].min()), "-", int(panel["year"].max()))

# =========================================================
# 5. BASIC DATA SUMMARY
# =========================================================

summary_cols = ["EAR", "ROA", "LLR", "LIQ", "WFS"]

print("\nSummary statistics for core indicators:")
print(panel[summary_cols].describe().T[["mean", "std", "min", "max"]])

print("\nMissing values in core indicators and lagged predictors:")
print(panel[
    ["EAR", "ROA", "LLR", "LIQ", "WFS",
     "EAR_lag1", "ROA_lag1", "LLR_lag1", "LIQ_lag1", "prev_y"]
].isna().sum())
# =========================================================
# 6. VALIDATE TARGET VARIABLES
# =========================================================

if TARGET_COL not in ["y_true", "y_alt"]:
    raise ValueError("TARGET_COL must be either 'y_true' or 'y_alt'.")
panel["prev_target"] = panel.groupby("country")[TARGET_COL].shift(1)

print("\nPrimary target distribution:")
print(panel["y_true"].value_counts(normalize=True).sort_index())

print("\nAlternative target distribution:")
print(panel["y_alt"].value_counts(normalize=True).sort_index())

print("\nStress score distribution:")
print(panel["stress_score"].value_counts().sort_index())


# =========================================================
# 7. BUILD LAGGED FEATURES
# =========================================================

main_features = ["EAR", "ROA", "LLR", "LIQ"]

print("\nFeatures used:", main_features)


feature_lags_main = [f"{c}_lag1" for c in main_features]

# =========================================================
#8. CORRELATION MATRIX AND VIF DIAGNOSTICS
# =========================================================

# Use lagged predictors
diag_data = panel[feature_lags_main].copy()

# Drop missing values
diag_data = diag_data.dropna().reset_index(drop=True)

# =========================================================
# 1. CORRELATION MATRIX
# =========================================================

corr_matrix = diag_data.corr()

print("\n=== CORRELATION MATRIX ===")
print(corr_matrix.round(3))

# Optional: save
corr_matrix.to_csv(
    os.path.join(OUTPUT_DIR, "correlation_matrix.csv")
)

# =========================================================
# 2. VARIANCE INFLATION FACTORS (VIF)
# =========================================================

X_vif = sm.add_constant(diag_data)

vif_table = pd.DataFrame({
    "Variable": X_vif.columns,
    "VIF": [
        variance_inflation_factor(X_vif.values, i)
        for i in range(X_vif.shape[1])
    ]
})

# Remove intercept row
vif_table = vif_table[vif_table["Variable"] != "const"]

print("\n=== VARIANCE INFLATION FACTORS ===")
print(vif_table.round(3).to_string(index=False))

# Optional: save
vif_table.to_csv(
    os.path.join(OUTPUT_DIR, "vif_diagnostics.csv"),
    index=False
)



# =========================================================
# 9. OOS DATES
# =========================================================

oos_dates = make_oos_dates(panel, warmup=WARMUP_YEARS)

if oos_dates.empty:
    raise RuntimeError("No country has enough years for out-of-sample evaluation.")

print("\nOOS dates sample:")
print(oos_dates.head())


# =========================================================
# 10. STRUCTURAL BASELINE
# =========================================================

def compute_structural_prob(
    df,
    country,
    year,
    target_col="y_true",
    alpha=0.5,
    k=2.0,
    eps=EPS,
):
    """
    Structural persistence model:
    EWMA of country-specific past distress, shrunk toward pooled historical prevalence.

    Uses only observations with year < forecast year.
    """
    hist_all = df[df["year"] < year].copy()

    if hist_all.empty:
        return 0.5

    global_prior = hist_all[target_col].mean()

    if pd.isna(global_prior):
        global_prior = 0.5

    hist_c = hist_all[hist_all["country"] == country].sort_values("year").copy()

    if hist_c.empty:
        return float(safe_prob(global_prior, eps=eps))

    ewma = hist_c[target_col].ewm(alpha=alpha, adjust=False).mean()

    n_hist = len(hist_c)
    lam = n_hist / (n_hist + k)

    m_last = ewma.iloc[-1]

    if pd.isna(m_last):
        m_last = global_prior

    p_struct = lam * m_last + (1.0 - lam) * global_prior

    return float(safe_prob(p_struct, eps=eps))


def structural_oos(df, oos_dates, target_col="y_true", alpha=0.5, k=2.0):
    rows = []

    for _, row in oos_dates.iterrows():
        c = row["country"]
        y_star = int(row["year"])

        test_row = df[(df["country"] == c) & (df["year"] == y_star)]

        if test_row.empty:
            continue

        p_struct = compute_structural_prob(
            df,
            c,
            y_star,
            target_col=target_col,
            alpha=alpha,
            k=k,
            eps=EPS,
        )

        rows.append(
            {
                "country": c,
                "year": y_star,
                "h": 1,
                target_col: int(test_row[target_col].iloc[0]),
                "p_struct": p_struct,
            }
        )

    return pd.DataFrame(rows)


# =========================================================
# 11. DISCRIMINATIVE LOGIT OOS
# =========================================================

def logit_oos_recursive_pooled(df, oos_dates, features, target_col="y_true"):
    """
    Pooled recursive ridge logit:
    For each forecast year, train on all countries with year < forecast year.
    """
    rows = []

    for _, row in oos_dates.iterrows():
        c = row["country"]
        y_star = int(row["year"])

        tr = df[df["year"] < y_star].copy()
        te = df[(df["country"] == c) & (df["year"] == y_star)].copy()

        if te.empty:
            continue

        tr_model = tr.dropna(subset=features + [target_col]).copy()
        ytr = tr_model[target_col].astype(int)

        if len(tr_model) < 8 or ytr.nunique() < 2:
            prior = tr[target_col].mean()

            if pd.isna(prior):
                prior = df[target_col].mean()

            if pd.isna(prior):
                prior = 0.5

            p = float(prior)

        else:
            pipe = make_logit_pipeline()
            pipe.fit(tr_model[features], ytr)
            p = float(pipe.predict_proba(te[features])[0, 1])

        rows.append(
            {
                "country": c,
                "year": int(y_star),
                "h": 1,
                "p_logit": float(safe_prob(p, eps=EPS)),
            }
        )

    return pd.DataFrame(rows)


def expanding_logit_coefficients(df, oos_dates, features, target_col="y_true"):
    """
    Coefficient paths using the same pipeline as the predictive logit.
    Coefficients are standardized-scale coefficients.
    """
    coef_rows = []

    for _, row in oos_dates.iterrows():
        y_star = int(row["year"])

        tr = df[df["year"] < y_star].copy()
        tr_model = tr.dropna(subset=features + [target_col]).copy()

        if len(tr_model) < 8:
            continue

        ytr = tr_model[target_col].astype(int)

        if ytr.nunique() < 2:
            continue

        pipe = make_logit_pipeline()
        pipe.fit(tr_model[features], ytr)

        coefs = pipe.named_steps["logisticregression"].coef_[0]

        coef_rows.append(
            {"forecast_year": y_star, **dict(zip(features, coefs))}
        )

    return pd.DataFrame(coef_rows)


# =========================================================
# 12. BUILD OOS TABLE
# =========================================================

oos_struct = structural_oos(
    panel,
    oos_dates,
    target_col=TARGET_COL,
    alpha=STRUCT_ALPHA,
    k=STRUCT_K,
)

oos_logit = logit_oos_recursive_pooled(
    panel,
    oos_dates,
    feature_lags_main,
    target_col=TARGET_COL,
)

oos = (
    oos_struct.merge(oos_logit, on=["country", "year", "h"], how="left")
    .sort_values(["country", "year"])
    .reset_index(drop=True)
)

oos = oos.merge(
    panel[["country", "year", "prev_target"]],
    on=["country", "year"],
    how="left",
)

oos["p_naive"] = [
    compute_naive_prob(panel, c, y, target_col=TARGET_COL)
    for c, y in zip(oos["country"], oos["year"])
]

oos["p_lag"] = [
    compute_lag_prob(panel, c, y, target_col=TARGET_COL)
    for c, y in zip(oos["country"], oos["year"])
]

# Standardize target column name for downstream plotting if needed.
if TARGET_COL != "y_true":
    oos["y_true"] = oos[TARGET_COL]

print("\nCheck benchmark equality:")
print("NAIVE equals LAG?", np.allclose(oos["p_naive"], oos["p_lag"], equal_nan=True))


# =========================================================
# 13. HYBRID FITTING
# =========================================================

def fit_offset_glm(lp_struct, lp_logit, y, alpha=1e-4):
    """
    Estimate hybrid logit:
    logit(p_H) = a + (1-w)logit(p_S) + w logit(p_D)

    Equivalent offset form:
    logit(p_H) = offset(logit(p_S)) + a + w[logit(p_D)-logit(p_S)]

    w is clipped to [0,1] to match the manuscript.
    """
    X = pd.DataFrame({"delta": lp_logit - lp_struct})
    X = sm.add_constant(X, has_constant="add")

    glm = sm.GLM(
        y.astype(int),
        X,
        family=sm.families.Binomial(),
        offset=lp_struct,
    )

    try:
        res = glm.fit_regularized(alpha=alpha, L1_wt=0.0)
        a = float(res.params["const"])
        w = float(res.params["delta"])
    except Exception:
        res = glm.fit()
        a = float(res.params["const"])
        w = float(res.params["delta"])

    return float(np.clip(w, 0.0, 1.0)), float(a)


def cross_fitted_hybrid(
    oos_df,
    target_col="y_true",
    use_isotonic=True,
    iso_blend=0.10,
    eps=EPS,
):
    """
    Leave-one-country-out hybrid fitting on OOS predictions.
    Calibration is supplementary and reported separately.
    """
    df = oos_df.dropna(subset=[target_col, "p_struct", "p_logit", "country"]).copy()

    df["p_struct"] = safe_prob(df["p_struct"].values, eps=eps)
    df["p_logit"] = safe_prob(df["p_logit"].values, eps=eps)

    if len(df) < 8 or df[target_col].nunique() < 2:
        df["p_hybrid_cv"] = 0.5 * df["p_struct"] + 0.5 * df["p_logit"]
        df["p_hybrid_cv_cal"] = df["p_hybrid_cv"]

        return {"mode": "avg_too_small", "a": 0.0, "w": 0.5}, df

    y = df[target_col].astype(int).values
    lp_struct = safe_logit(df["p_struct"].values, eps)
    lp_logit = safe_logit(df["p_logit"].values, eps)

    df["p_hybrid_cv"] = np.nan

    for c in df["country"].unique():
        te = df["country"].values == c
        tr = ~te

        if tr.sum() < 6 or np.unique(y[tr]).size < 2:
            p_te = np.clip(
                0.5 * df.loc[te, "p_struct"].values
                + 0.5 * df.loc[te, "p_logit"].values,
                eps,
                1.0 - eps,
            )
            df.loc[te, "p_hybrid_cv"] = p_te
            continue

        w, a = fit_offset_glm(lp_struct[tr], lp_logit[tr], y[tr], alpha=1e-4)

        z_te = a + w * (lp_logit[te] - lp_struct[te]) + lp_struct[te]
        p_te = sigmoid(z_te)

        df.loc[te, "p_hybrid_cv"] = safe_prob(p_te, eps=eps)

    if use_isotonic and df["p_hybrid_cv"].nunique() > 2 and df[target_col].nunique() > 1:
        df["p_hybrid_cv_cal"] = np.nan

        for c in df["country"].unique():
            te = df["country"].values == c
            tr = ~te

            if (
                tr.sum() < 8
                or np.unique(y[tr]).size < 2
                or df.loc[tr, "p_hybrid_cv"].nunique() < 3
            ):
                df.loc[te, "p_hybrid_cv_cal"] = df.loc[te, "p_hybrid_cv"].values
                continue

            iso = IsotonicRegression(out_of_bounds="clip")
            iso.fit(df.loc[tr, "p_hybrid_cv"].values, df.loc[tr, target_col].values)

            p_iso = iso.transform(df.loc[te, "p_hybrid_cv"].values)

            df.loc[te, "p_hybrid_cv_cal"] = np.clip(
                (1.0 - iso_blend) * p_iso
                + iso_blend * df.loc[te, "p_hybrid_cv"].values,
                eps,
                1.0 - eps,
            )
    else:
        df["p_hybrid_cv_cal"] = df["p_hybrid_cv"]

    w_all, a_all = fit_offset_glm(lp_struct, lp_logit, y, alpha=1e-4)

    return {"mode": "offset_glm_cv", "a": a_all, "w": w_all}, df


fit_stack, oos_preds = cross_fitted_hybrid(
    oos,
    target_col=TARGET_COL,
    use_isotonic=USE_ISOTONIC,
    iso_blend=ISO_BLEND,
)


# =========================================================
# 14. RESULTS
# =========================================================

results_table = compare_models(
    oos_preds,
    target_col=TARGET_COL,
    include_calibrated=False,
)

results_table_with_cal = compare_models(
    oos_preds,
    target_col=TARGET_COL,
    include_calibrated=True,
)

print("\n=== MAIN MODEL COMPARISON ===")
print(results_table.to_string(index=False))

print("\n=== SUPPLEMENTARY MODEL COMPARISON WITH CALIBRATED HYBRID ===")
print(results_table_with_cal.to_string(index=False))

print("\n=== HYBRID PARAMETERS - FULL OOS FIT FOR REPORTING ===")
print(fit_stack)


# =========================================================
# 15. BOOTSTRAP AUC COMPARISONS
# =========================================================

print("\n=== POOLED BOOTSTRAP AUC COMPARISONS ===")

y = oos_preds[TARGET_COL].values.astype(int)

comparisons = [
    ("p_struct", "p_logit", "STRUCT vs LOGIT"),
    ("p_struct", "p_hybrid_cv", "STRUCT vs HYBRID"),
    ("p_logit", "p_hybrid_cv", "LOGIT vs HYBRID"),
    ("p_naive", "p_lag", "NAIVE vs LAG"),
]

boot_rows = []

for c1, c2, label in comparisons:
    mask = oos_preds[[c1, c2, TARGET_COL]].notna().all(axis=1).values

    if mask.sum() < 3 or len(np.unique(y[mask])) < 2:
        print(label, "Not enough valid observations/classes.")
        continue

    res = bootstrap_auc_diff(
        y[mask],
        oos_preds.loc[mask, c1].values.astype(float),
        oos_preds.loc[mask, c2].values.astype(float),
        n_boot=1000,
        random_state=RANDOM_STATE,
    )

    boot_rows.append(
        {
            "Comparison": label,
            "Mean Diff.": res["mean_diff"],
            "Lower CI": res["ci_lower"],
            "Upper CI": res["ci_upper"],
            "n_boot_used": res["n_boot_used"],
        }
    )

    print(label, res)

bootstrap_table = pd.DataFrame(boot_rows)


# =========================================================
# 16. COEFFICIENT PATHS
# =========================================================

coef_df = expanding_logit_coefficients(
    panel,
    oos_dates,
    feature_lags_main,
    target_col=TARGET_COL,
)

if not coef_df.empty:
    coef_summary = (
        coef_df[feature_lags_main]
        .agg(["mean", "std"])
        .T
        .reset_index()
        .rename(columns={"index": "Variable", "mean": "Mean", "std": "Std. Dev."})
    )

    coef_summary["Sign"] = np.where(coef_summary["Mean"] >= 0, "+", "-")
else:
    coef_summary = pd.DataFrame(columns=["Variable", "Mean", "Std. Dev.", "Sign"])




# =========================================================
# 17. ROBUSTNESS CHECK: BINNING + WEIGHT OF EVIDENCE (WoE) LOGIT
# =========================================================

def make_quantile_bins(x, n_bins=4):
    """
    Quantile binning with duplicate-edge handling.
    """
    return pd.qcut(x, q=n_bins, duplicates="drop")

def compute_woe_map(x_binned, y, eps=0.5):
    """
    Computes WoE per bin.
    WoE = log( non-distress share / distress share )
    """
    tmp = pd.DataFrame({"bin": x_binned, "y": y}).dropna()

    total_good = (tmp["y"] == 0).sum()
    total_bad = (tmp["y"] == 1).sum()

    woe_map = {}

    for b, g in tmp.groupby("bin", observed=False):
        good = (g["y"] == 0).sum()
        bad = (g["y"] == 1).sum()

        good_share = (good + eps) / (total_good + eps)
        bad_share = (bad + eps) / (total_bad + eps)

        woe_map[b] = np.log(good_share / bad_share)

    return woe_map

def apply_woe(train_x, train_y, test_x, n_bins=4):
    """
    Fits bins and WoE on training data only, then applies to test data.
    """
    train_bins = make_quantile_bins(train_x, n_bins=n_bins)

    # extract bin edges from training qcut
    categories = train_bins.cat.categories

    if len(categories) < 2:
        return (
            pd.Series(np.zeros(len(train_x)), index=train_x.index),
            pd.Series(np.zeros(len(test_x)), index=test_x.index),
        )

    edges = [categories[0].left] + [c.right for c in categories]
    edges[0] = -np.inf
    edges[-1] = np.inf

    train_bins = pd.cut(train_x, bins=edges, include_lowest=True)
    test_bins = pd.cut(test_x, bins=edges, include_lowest=True)

    woe_map = compute_woe_map(train_bins, train_y)

    train_woe = train_bins.map(woe_map).astype(float)
    test_woe = test_bins.map(woe_map).astype(float)

    train_woe = train_woe.fillna(0.0)
    test_woe = test_woe.fillna(0.0)

    return train_woe, test_woe

def recursive_woe_logit_oos(df, oos_dates, features, target_col="y_true", n_bins=4):
    rows = []
    coef_rows = []

    for _, row in oos_dates.iterrows():
        c = row["country"]
        y_star = int(row["year"])

        tr = df[df["year"] < y_star].copy()
        te = df[(df["country"] == c) & (df["year"] == y_star)].copy()

        if te.empty:
            continue

        tr_model = tr.dropna(subset=features + [target_col]).copy()
        te_model = te.copy()

        ytr = tr_model[target_col].astype(int)

        if len(tr_model) < 8 or ytr.nunique() < 2:
            prior = tr[target_col].mean()
            if pd.isna(prior):
                prior = df[target_col].mean()
            if pd.isna(prior):
                prior = 0.5

            rows.append({
                "country": c,
                "year": y_star,
                "h": 1,
                target_col: int(te[target_col].iloc[0]),
                "p_logit_woe": float(np.clip(prior, 1e-6, 1 - 1e-6)),
            })
            continue

        Xtr_woe = pd.DataFrame(index=tr_model.index)
        Xte_woe = pd.DataFrame(index=te_model.index)

        for f in features:
            train_woe, test_woe = apply_woe(
                tr_model[f],
                ytr,
                te_model[f],
                n_bins=n_bins
            )

            Xtr_woe[f"{f}_woe"] = train_woe
            Xte_woe[f"{f}_woe"] = test_woe

        model = make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            LogisticRegression(
                penalty="l2",
                solver="lbfgs",
                max_iter=1000,
                random_state=RANDOM_STATE
            )
        )

        model.fit(Xtr_woe, ytr)
        p = model.predict_proba(Xte_woe)[0, 1]

        rows.append({
            "country": c,
            "year": y_star,
            "h": 1,
            target_col: int(te[target_col].iloc[0]),
            "p_logit_woe": float(np.clip(p, 1e-6, 1 - 1e-6)),
        })

        coefs = model.named_steps["logisticregression"].coef_[0]
        coef_rows.append({
            "forecast_year": y_star,
            **dict(zip(Xtr_woe.columns, coefs))
        })

    return pd.DataFrame(rows), pd.DataFrame(coef_rows)

# Run WoE recursive OOS logit
oos_woe, coef_woe = recursive_woe_logit_oos(
    panel,
    oos_dates,
    feature_lags_main,
    target_col=TARGET_COL,
    n_bins=4
)

# Merge with your existing OOS predictions
oos_compare = oos_preds.merge(
    oos_woe[["country", "year", "p_logit_woe"]],
    on=["country", "year"],
    how="left"
)

def evaluate_single_model(df, y_col, p_col, label):
    tmp = df[[y_col, p_col]].dropna().copy()
    y = tmp[y_col].astype(int).values
    p = np.clip(tmp[p_col].astype(float).values, 1e-6, 1 - 1e-6)

    return {
        "Model": label,
        "Brier": brier_score_loss(y, p),
        "LogLoss": log_loss(y, p, labels=[0, 1]),
        "AUC": roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan,
        "AP": average_precision_score(y, p) if len(np.unique(y)) > 1 else np.nan,
        "N": len(y)
    }

robustness_results = pd.DataFrame([
    evaluate_single_model(oos_compare, TARGET_COL, "p_logit", "Raw continuous logit"),
    evaluate_single_model(oos_compare, TARGET_COL, "p_logit_woe", "WoE-binned logit"),
    evaluate_single_model(oos_compare, TARGET_COL, "p_struct", "Structural baseline"),
    evaluate_single_model(oos_compare, TARGET_COL, "p_hybrid_cv", "Hybrid")
])

print("\n=== WoE ROBUSTNESS RESULTS ===")
print(robustness_results.to_string(index=False))

print("\n=== WoE COEFFICIENT SUMMARY ===")
if not coef_woe.empty:
    coef_woe_summary = (
        coef_woe.drop(columns=["forecast_year"])
        .agg(["mean", "std"])
        .T
        .reset_index()
        .rename(columns={"index": "Variable", "mean": "Mean", "std": "Std. Dev."})
    )
    coef_woe_summary["Sign"] = np.where(coef_woe_summary["Mean"] >= 0, "+", "-")
    print(coef_woe_summary.to_string(index=False))
else:
    print("No WoE coefficients estimated.")




# =========================================================
# 18. SAVE TABLE OUTPUTS
# =========================================================

oos_preds.to_csv(os.path.join(OUTPUT_DIR, "oos_predictions_hybrid.csv"), index=False)
results_table.to_csv(os.path.join(OUTPUT_DIR, "results_table_main.csv"), index=False)
results_table_with_cal.to_csv(os.path.join(OUTPUT_DIR, "results_table_with_calibration.csv"), index=False)
bootstrap_table.to_csv(os.path.join(OUTPUT_DIR, "bootstrap_auc_comparisons.csv"), index=False)
coef_df.to_csv(os.path.join(OUTPUT_DIR, "recursive_logit_coefficients.csv"), index=False)
coef_summary.to_csv(os.path.join(OUTPUT_DIR, "logit_coefficient_summary.csv"), index=False)
oos_compare.to_csv(os.path.join(OUTPUT_DIR, "oos_predictions_with_woe_logit.csv"), index=False)
robustness_results.to_csv(os.path.join(OUTPUT_DIR, "woe_robustness_results.csv"), index=False)
coef_woe.to_csv(os.path.join(OUTPUT_DIR, "woe_recursive_logit_coefficients.csv"), index=False)

if not coef_woe.empty:
    coef_woe_summary.to_csv(os.path.join(OUTPUT_DIR, "woe_coefficient_summary.csv"), index=False)

print("\nSaved table outputs to:", OUTPUT_DIR)


# =========================================================
# 19. PLOTS
# =========================================================

plot_target = TARGET_COL

fig, ax = plt.subplots(figsize=(5.5, 5))

roc_models = {
    "Structural baseline (EWMA)": "p_struct",
    "Discriminative logit": "p_logit",
    "Hybrid": "p_hybrid_cv",
    "Naive prevalence": "p_naive",
    "Lag only": "p_lag",
}

for label, col in roc_models.items():
    y_plot, p_plot = valid_xy(oos_preds, plot_target, col)

    if len(y_plot) == 0 or len(np.unique(y_plot)) < 2:
        continue

    fpr, tpr, _ = roc_curve(y_plot, p_plot)
    ax.plot(
        fpr,
        tpr,
        linewidth=1.8,
        label=f"{label} (AUC = {auc(fpr, tpr):.3f})",
    )

ax.plot([0, 1], [0, 1], "k--", linewidth=1)
ax.set_title("ROC curves (pooled out-of-sample)", fontsize=10)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.legend(loc="lower right", fontsize=7)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "roc_curves.png"), dpi=300, bbox_inches="tight")
plt.close(fig)


fig, ax = plt.subplots(figsize=(5.5, 5))

pr_models = {
    "Structural baseline (EWMA)": "p_struct",
    "Discriminative logit": "p_logit",
    "Hybrid": "p_hybrid_cv",
    "Naive prevalence": "p_naive",
    "Lag only": "p_lag",
}

for label, col in pr_models.items():
    y_plot, p_plot = valid_xy(oos_preds, plot_target, col)

    if len(y_plot) == 0 or len(np.unique(y_plot)) < 2:
        continue

    precision, recall, _ = precision_recall_curve(y_plot, p_plot)
    ap = average_precision_score(y_plot, p_plot)

    ax.plot(
        recall,
        precision,
        linewidth=1.8,
        label=f"{label} (AP = {ap:.3f})",
    )

baseline = oos_preds[plot_target].mean()

ax.hlines(
    baseline,
    0,
    1,
    linestyles="--",
    linewidth=1,
    label=f"Baseline = {baseline:.3f}",
)

ax.set_title("Precision-recall curves", fontsize=10)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "precision_recall_curves.png"), dpi=300, bbox_inches="tight")
plt.close(fig)

# =========================================================
# PROBABILITY DISTRIBUTION PLOT 
# =========================================================

dist_models = {
    "Structural": "p_struct",
    "Hybrid": "p_hybrid_cv",
    "Logit": "p_logit",
}

fig, axes = plt.subplots(
    1,
    3,
    figsize=(12, 4),
    sharey=True
)

for ax, (label, col) in zip(axes, dist_models.items()):

    tmp = oos_preds[[TARGET_COL, col]].dropna().copy()

    if tmp.empty:
        continue

    no_distress = tmp[tmp[TARGET_COL] == 0]

    if not no_distress.empty:
        sns.kdeplot(
            data=no_distress,
            x=col,
            fill=True,
            alpha=0.4,
            label="No distress",
            ax=ax,
            cut=0,
            clip=(0, 1),
            linewidth=1.5
        )

    distress = tmp[tmp[TARGET_COL] == 1]

    if not distress.empty:
        sns.kdeplot(
            data=distress,
            x=col,
            fill=True,
            alpha=0.4,
            label="Distress",
            ax=ax,
            cut=0,
            clip=(0, 1),
            linewidth=1.5
        )

    ax.set_title(label, fontsize=11)
    ax.set_xlabel("Predicted probability", fontsize=10)
    ax.set_xlim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

axes[0].set_ylabel("Density", fontsize=10)

fig.suptitle(
    "Predicted Probability Distributions by Realized Distress Outcomes",
    fontsize=12
)

plt.tight_layout()

dist_path = os.path.join(
    FIG_DIR,
    "probability_distributions.png"
)

plt.savefig(
    dist_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)

print(f"Probability distribution figure saved to: {dist_path}")


# =========================================================
# APPENDIX: CALIBRATION CURVE 
# =========================================================

cal_models = {
    "Structural": "p_struct",
    "Hybrid": "p_hybrid_cv",
}

fig, ax = plt.subplots(figsize=(6, 5))

for label, col in cal_models.items():

    tmp = oos_preds[[TARGET_COL, col]].dropna().copy()

    if tmp.empty:
        continue

    y_true = tmp[TARGET_COL].astype(int).values
    probs = tmp[col].astype(float).values

    if len(np.unique(y_true)) < 2:
        continue

    prob_true, prob_pred = calibration_curve(
        y_true,
        probs,
        n_bins=5,
        strategy="quantile"
    )

    ax.plot(
        prob_pred,
        prob_true,
        marker="o",
        linewidth=2,
        label=label
    )

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.5,
    label="Perfect calibration"
)

ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed distress frequency")
ax.set_title("Calibration Curve: Predicted and Observed Distress Probabilities")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()

cal_path = os.path.join(
    FIG_DIR,
    "appendix_calibration_curve.png"
)

plt.savefig(
    cal_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)

print(f"Calibration curve saved to: {cal_path}")


# =========================================================
# RECURSIVE COEFFICIENT PATHS (APPENDIX)
# =========================================================

if not coef_df.empty:

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(10, 7),
        sharex=True
    )

    axes = axes.flatten()

    feature_labels = {
        "EAR_lag1": r"EAR$_{t-1}$",
        "ROA_lag1": r"ROA$_{t-1}$",
        "LLR_lag1": r"LLR$_{t-1}$",
        "LIQ_lag1": r"LIQ$_{t-1}$",
    }

    feature_colors = {
        "EAR_lag1": "tab:blue",
        "ROA_lag1": "tab:orange",
        "LLR_lag1": "tab:green",
        "LIQ_lag1": "tab:red",
    }

    for ax, col in zip(axes, feature_lags_main):

        if col not in coef_df.columns:
            ax.set_visible(False)
            continue

        ax.plot(
            coef_df["forecast_year"],
            coef_df[col],
            marker="o",
            linewidth=2,
            color=feature_colors.get(col, "tab:blue"),
        )

        ax.axhline(
            0,
            linestyle="--",
            linewidth=1,
            color="black",
        )

        ax.set_title(
            f"{feature_labels.get(col, col)} coefficient path",
            fontsize=10,
        )

        ax.set_xlabel("Forecast year")
        ax.set_ylabel("Coefficient")
        ax.grid(alpha=0.3)

    fig.suptitle(
        "Recursive Coefficient Paths for Discriminative Model",
        fontsize=12,
    )

    plt.tight_layout()

    coef_path = os.path.join(
        FIG_DIR,
        "appendix_coefficient_paths.png"
    )

    plt.savefig(
        coef_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)

    print(f"Coefficient paths figure saved to: {coef_path}")

else:
    print("Coefficient dataframe is empty.")

# =========================================================
# COUNTRY-LEVEL PROBABILITY TRAJECTORIES
# =========================================================

countries = sorted(oos_preds["country"].dropna().unique())

n_cols = 2
n_rows = int(np.ceil(len(countries) / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(10, 3.2 * n_rows),
    sharex=False,
    sharey=True
)

axes = axes.flatten()

for ax, country in zip(axes, countries):

    tmp = (
        oos_preds[oos_preds["country"] == country]
        .sort_values("year")
        .copy()
    )

    # Ensure integer years
    tmp["year"] = tmp["year"].astype(int)

    ax.plot(
        tmp["year"],
        tmp["p_struct"],
        marker="o",
        linewidth=1.8,
        label="Structural"
    )

    ax.plot(
        tmp["year"],
        tmp["p_hybrid_cv"],
        marker="o",
        linewidth=1.8,
        label="Hybrid"
    )

    ax.plot(
        tmp["year"],
        tmp["p_logit"],
        marker="o",
        linewidth=1.8,
        label="Logit"
    )

    distress_years = tmp.loc[tmp[TARGET_COL] == 1, "year"]

    if not distress_years.empty:
        ax.scatter(
            distress_years,
            np.ones(len(distress_years)),
            marker="x",
            s=45,
            label="Realized distress"
        )

    ax.set_title(country, fontsize=10)
    ax.set_xlabel("Year", fontsize=8)
    ax.set_ylabel("Probability", fontsize=8)
    ax.set_ylim(0, 1.05)

    # Integer year ticks only
    ax.set_xticks(tmp["year"].unique())
    ax.set_xticklabels(tmp["year"].unique().astype(int))

    ax.grid(alpha=0.3)

    ax.tick_params(
        axis="x",
        rotation=45,
        labelsize=7
    )

    ax.tick_params(
        axis="y",
        labelsize=7
    )

# Remove unused axes
for ax in axes[len(countries):]:
    ax.remove()

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=4,
    fontsize=8,
    frameon=False
)

fig.suptitle(
    "Predicted Probability and Realized Distress by Country",
    fontsize=12
)

plt.tight_layout(rect=[0, 0.05, 1, 0.95])

country_path = os.path.join(
    FIG_DIR,
    "country_probability_trajectories.png"
)

plt.savefig(
    country_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)

print(f"Country-level probability trajectories saved to: {country_path}")


# =========================================================
# POOLED PROBABILITY SIGNALS OVER TIME (APPENDIX)
# =========================================================

# Ensure integer year
oos_preds["year"] = oos_preds["year"].astype(int)

# Aggregate yearly averages
pooled = (
    oos_preds.groupby("year", as_index=False)
    .agg({
        "p_struct": "mean",
        "p_hybrid_cv": "mean",
        "p_logit": "mean",
        TARGET_COL: "mean",
    })
    .sort_values("year")
)

# Ensure integer years
pooled["year"] = pooled["year"].astype(int)

fig, ax = plt.subplots(figsize=(8, 5))

# Structural
ax.plot(
    pooled["year"],
    pooled["p_struct"],
    marker="o",
    linewidth=2,
    label="Structural"
)

# Hybrid
ax.plot(
    pooled["year"],
    pooled["p_hybrid_cv"],
    marker="o",
    linewidth=2,
    label="Hybrid"
)

# Logit
ax.plot(
    pooled["year"],
    pooled["p_logit"],
    marker="o",
    linewidth=2,
    label="Logit"
)

# Realized distress frequency
ax.plot(
    pooled["year"],
    pooled[TARGET_COL],
    marker="x",
    linestyle="--",
    linewidth=1.5,
    markersize=7,
    label="Realized distress frequency"
)

# Force integer year ticks
ax.set_xticks(pooled["year"].unique())
ax.set_xticklabels(
    pooled["year"].unique().astype(int),
    rotation=45
)

ax.set_xlabel("Year")
ax.set_ylabel("Average probability / distress frequency")

ax.set_title(
    "Pooled Probability Signals and Realized Distress Over Time"
)

ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)

ax.legend(
    fontsize=8,
    loc="best"
)

plt.tight_layout()

pooled_path = os.path.join(
    FIG_DIR,
    "appendix_pooled_probability_signals.png"
)

plt.savefig(
    pooled_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)

print(f"Pooled probability signals figure saved to: {pooled_path}")

# =========================================================
# SAVE BOOTSTRAP MEAN DIFFERENCE TABLE
# =========================================================

# Round for publication formatting
bootstrap_table_final = bootstrap_table.copy()

bootstrap_table_final["Mean Diff."] = (
    bootstrap_table_final["Mean Diff."].round(3)
)

bootstrap_table_final["Lower CI"] = (
    bootstrap_table_final["Lower CI"].round(3)
)

bootstrap_table_final["Upper CI"] = (
    bootstrap_table_final["Upper CI"].round(3)
)

print("\n=== FINAL BOOTSTRAP AUC DIFFERENCE TABLE ===")
print(bootstrap_table_final.to_string(index=False))

# Save CSV
bootstrap_path = os.path.join(
    OUTPUT_DIR,
    "bootstrap_auc_mean_differences.csv"
)

bootstrap_table_final.to_csv(
    bootstrap_path,
    index=False
)

print(f"Bootstrap mean-difference table saved to: {bootstrap_path}")

print("\nReplication script completed successfully.")

Loaded replication dataset: cleaned_full_panel.csv
Panel shape: (125, 22)
Countries: ['Botswana', 'Malawi', 'Namibia', 'South Africa', 'Tanzania', 'Zambia']
Years: 2004 - 2024

Summary statistics for core indicators:
         mean       std       min       max
EAR  0.219986  0.277813  0.000000  1.000000
ROA  0.035614  0.073593 -0.035263  0.572277
LLR  0.000223  0.002115  0.000000  0.020065
LIQ  0.208084  0.243993  0.000000  1.285858
WFS  0.105668  0.205094  0.000000  1.990363

Missing values in core indicators and lagged predictors:
EAR         16
ROA         10
LLR         35
LIQ         35
WFS         16
EAR_lag1    21
ROA_lag1    16
LLR_lag1    39
LIQ_lag1    39
prev_y       6
dtype: int64

Primary target distribution:
y_true
0    0.76
1    0.24
Name: proportion, dtype: float64

Alternative target distribution:
y_alt
0    0.256
1    0.744
Name: proportion, dtype: float64

Stress score distribution:
stress_score
0    32
1    63
2    26
3     4
Name: count, dtype: int64

Features used